In [1]:
# ==================================================
# CLEAN MINIMAL INSTALL FOR LEGAL RAG PROJECT
# ==================================================
!pip install -q \
sentence-transformers \
transformers \
faiss-cpu \
gradio \
pandas \
numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.9 MB/s eta 0:00:00


In [2]:
# ==========================================================
# CELL 1 : INSTALL REQUIRED LIBRARIES (Google Colab)
# ==========================================================
!pip install -q sentence-transformers transformers faiss-cpu gradio pandas numpy

In [3]:
# ==========================================================
# CELL 2 : IMPORT LIBRARIES
# ==========================================================
import os
import json
import zipfile
import pandas as pd
import numpy as np
import faiss
import gradio as gr

from sentence_transformers import SentenceTransformer
from transformers import pipeline

In [ ]:
# ==========================================================
# CELL 3 : UPLOAD ZIP DATASET
# Upload your data.zip file here
# ==========================================================
from google.colab import files

uploaded = files.upload()

In [ ]:
# ==========================================================
# CELL 4 : EXTRACT ZIP FILE
# ==========================================================
zip_path = "data.zip"
extract_path = "dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("ZIP Extracted Successfully")
print(os.listdir(extract_path))

In [ ]:
# ==========================================================
# CELL 5 : LOAD JSON DATASET
# ==========================================================
json_path = "dataset/CUADv1.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

contracts = data["data"]

print("Total Contracts:", len(contracts))

In [ ]:
# ==========================================================
# CELL 6 : EXTRACT TEXT DOCUMENTS
# ==========================================================
documents = []

for contract in contracts:
    title = contract["title"]

    for para in contract["paragraphs"]:
        documents.append({
            "title": title,
            "text": para["context"]
        })

print("Documents Loaded:", len(documents))

In [ ]:
# ==========================================================
# CELL 7 : TEXT CHUNKING (Without LangChain)
# ==========================================================
def split_text(text, chunk_size=500, overlap=100):

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks


chunks = []

for doc in documents:

    parts = split_text(doc["text"])

    for chunk in parts:
        chunks.append({
            "title": doc["title"],
            "chunk": chunk
        })

print("Total Chunks:", len(chunks))

In [ ]:
# ==========================================================
# CELL 8 : LOAD EMBEDDING MODEL
# ==========================================================
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
# ==========================================================
# CELL 9 : GENERATE EMBEDDINGS
# ==========================================================
texts = [row["chunk"] for row in chunks]

embeddings = embed_model.encode(
    texts,
    show_progress_bar=True
)

print("Embeddings Shape:", embeddings.shape)

In [ ]:
# ==========================================================
# CELL 10 : CREATE FAISS VECTOR DATABASE
# ==========================================================
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))

print("FAISS Index Created")

In [ ]:
# ==========================================================
# CELL 11 : RETRIEVAL FUNCTION
# ==========================================================
def retrieve_docs(query, top_k=3):

    query_embedding = embed_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx]["chunk"])

    return results

In [ ]:
# ==========================================================
# CELL 12 : TEST SEARCH
# ==========================================================
query = "termination clause"

docs = retrieve_docs(query)

for doc in docs:
    print(doc[:1000])
    print("="*80)

In [ ]:
# ==========================================================
# FIXED CELL 13 : LOAD LLM
# New transformers version uses text-generation
# ==========================================================
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="google/flan-t5-base",
    tokenizer="google/flan-t5-base"
)

print("Model Loaded Successfully")

In [ ]:
# ==========================================================
# FIXED CELL 14 : LEGAL RISK ANALYSIS FUNCTION
# ==========================================================
def analyze_contract(query):

    docs = retrieve_docs(query, top_k=3)
    context = "\n".join(docs)

    prompt = f"""
You are a legal AI assistant.

Analyze the contract clauses below.

User Query: {query}

Contract Text:
{context}

Return:
1. Risk Level
2. Explanation
3. Important Clause
4. Recommendation
"""

    output = generator(
        prompt,
        max_new_tokens=200,
        do_sample=False
    )

    return output[0]["generated_text"]

In [ ]:
# ==========================================================
# CELL 15 : FINAL TEST
# ==========================================================
question = "Find risky termination and payment clauses"

answer = analyze_contract(question)

print(answer)

In [ ]:
# ==========================================================
# CELL 16 : GRADIO WEB APP
# ==========================================================
def chatbot(query):
    return analyze_contract(query)

demo = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Enter legal query..."
    ),
    outputs=gr.Textbox(
        lines=10,
        label="Analysis Result"
    ),
    title="Legal RAG Contract Risk Analyzer",
    description="Analyze risky legal clauses using RAG + AI"
)

demo.launch(share=True)

In [ ]:
# ==========================================================
# CELL : DOWNLOAD MODEL LOCALLY IN GOOGLE COLAB
# FLAN-T5 BASE
# ==========================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
save_path = "/content/flan_t5_base_model"

# Download Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Download Model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Save Locally
tokenizer.save_pretrained(save_path)
model.save_pretrained(save_path)

print("Model Downloaded and Saved at:", save_path)

In [ ]:
# ==========================================================
# FIX: LOAD DOWNLOADED FLAN MODEL IN NEW TRANSFORMERS
# ==========================================================
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model="/content/flan_t5_base_model",
    tokenizer="/content/flan_t5_base_model"
)

print("Local Model Loaded Successfully")